## Exploratory Data Analysis

In this notebook, the Fashionpedia dataset is analyzed exhaustively. Note that the analysis is performed locally and not in Google Colab.

In [46]:
import json
import itertools
import collections
import numpy as np
from matplotlib import pyplot as plt

upto=27

with open('../OR_task2/coco/annotations/instances_attributes_train2020.json', 'r') as file:
    data_train = json.load(file)
    
with open('../OR_task2/coco/annotations/instances_attributes_val2020.json', 'r') as file:
    data_val = json.load(file)

with open('../OR_task2/coco/annotations/info_test2020.json', 'r') as file:
    infos = json.load(file)

In [47]:
id_filename_train = [[entry["id"],entry["file_name"],entry["width"],entry["height"]] for entry in data_train["images"]]
id_filename_val = [[entry["id"],entry["file_name"],entry["width"],entry["height"]] for entry in data_val["images"]]
metrics_images_train = {}
for i in id_filename_train:
    metrics_images_train[i[0]] = i[1:]
metrics_images_val = {}
for i in id_filename_val:
    metrics_images_val[i[0]] = i[1:]  

In [48]:
#Average size and aspect ratio of train and val images
dicts = [metrics_images_train,metrics_images_val]
for d in dicts:
    values = list(d.values())
    w = []
    h = []
    s = []
    ar = []
    for a in values:
        w.append(a[1])
        h.append(a[2])
        s.append(a[1]*a[2])
        ar.append(a[1]*1.0/(1.0*a[2]))
    print("W",np.mean(w))
    print("H",np.mean(h))
    print("S",np.mean(s))
    print("AR",np.mean(ar))
    print()

W 755.2479670341713
H 986.5135567586524
S 734987.8003638516
AR 0.7926327412039755

W 750.7392055267703
H 962.3765112262522
S 705654.493955095
AR 0.8241621427407255



In [49]:
categories = [x["name"] for x in infos["categories"]]
clean_cats = []
for category in categories: ##debloat
    if "," in category:
        clean_cats.append(category.split(",")[0])
    else:
        clean_cats.append(category)

In [ ]:
datas = [data_train, data_val]
inf = ["Train dataset","Val dataset"]
counts = []
for k in range(2):
    data = datas[k]
    #PROCESSEM PRIMER TOTES LES ANNOTATIONS
    annotated_images = []
    for entry in data["annotations"]:
        annotated_images.append(entry["image_id"])
    annotated_images = list(set(annotated_images))
    print("Total annotated images",len(annotated_images))
    annotations = {}    
    for i in annotated_images:
        annotations[i] = []
    for entry in data["annotations"]:
        annotations[entry["image_id"]].append(entry)
    anns_x_image=[]
    for key,annon in annotations.items():
        anns_x_image.append(len(annon))
    print(inf[k])
    print("All categories")
    print("total",np.sum(anns_x_image))
    print("mean",np.mean(anns_x_image))
    print("std",np.std(anns_x_image))

    annotations = {}    
    for i in annotated_images:
        annotations[i] = []
    for entry in data["annotations"]:
        if entry["category_id"]>=27: continue
        annotations[entry["image_id"]].append(entry)
    anns_x_image=[]
    for key,annon in annotations.items():
        anns_x_image.append(len(annon))
    print("Reduced categories")
    print("total",np.sum(anns_x_image))
    print("mean",np.mean(anns_x_image))
    print("std",np.std(anns_x_image))
    
    anns_cats = {}    
    for i in annotated_images:
        anns_cats[i] = []
    for entry in data["annotations"]:
        if entry["category_id"]>=27: continue
        anns_cats[entry["image_id"]].append(entry["category_id"])
    
    #coexistance matrix
    cross_categories = list(anns_cats.values())
    coexistance_matrix = np.zeros((upto, upto))
    for elem in cross_categories:
        for i in elem:
            for j in elem:
                coexistance_matrix[i][j]+=1
    
    plt.imshow(coexistance_matrix, cmap='hot', interpolation='nearest')
    plt.title("Coexistance matrix. "+str(inf[k]))
    plt.xticks(range(upto), labels = clean_cats[:27], rotation=90)
    plt.yticks(range(upto), labels = clean_cats[:27], rotation=0)
    plt.show()
    
    chain = itertools.chain(*cross_categories)
    freq_counter = collections.Counter(chain)
    D = dict(freq_counter)
    D = collections.OrderedDict(sorted(D.items()))
    counts.append(D)
    f = plt.figure()
    ax = f.add_subplot(111)
    ax.yaxis.tick_right()

    plt.barh(range(len(D)), list(D.values()), align='center')
    plt.yticks(range(len(D)), labels = clean_cats[:27], rotation=0)
    plt.ylim(-0.5,27-.5)
    ax.yaxis.set_inverted(True)
    plt.title("Total instances per category. "+str(inf[k]))
    plt.show()
    print()

In [ ]:
train_ratios = []
for c in counts[0]:
    train_ratios.append(counts[0][c]/45623)
val_ratios = []
for c in counts[1]:
    val_ratios.append(counts[1][c]/1158)
plt.bar(x=clean_cats[:27], height=100*(np.array(val_ratios) - np.array(train_ratios)))
plt.xticks(rotation=90)
plt.title("Representation difference (flat%) between Validation and Train")
plt.ylim(-6,38)
plt.xlim(-0.5,upto-0.5)
plt.show()

In [ ]:
def shoelace_formula(x,y):
    correction = x[-1] * y[0] - y[-1]* x[0]
    main_area = np.dot(x[:-1], y[1:]) - np.dot(y[:-1], x[1:])
    return 0.5*np.abs(main_area + correction)

def bboxarea(bbox):
    a = (bbox[2]*bbox[3])
    return np.abs(a)

In [ ]:
datas = [data_train, data_val]
inf = ["Train dataset","Val dataset"]
z = 0

for z in range(2):
    data = datas[z]
    info = inf[z]
    area_values = {}
    size_ratio_values = {}
    fragments = {}

    for clean_cat in clean_cats[:27]:
        area_values[clean_cat] = []
        fragments[clean_cat] = []
        size_ratio_values[clean_cat] = []
    #print(area_values)

    for annot in data["annotations"]:
        category_id = annot["category_id"]
        if category_id>=27: continue
        
        category = clean_cats[category_id]
        
        #potentially corrupt files
        if type(annot["segmentation"])!=type([]):
            #print(annot["segmentation"])
            continue
        #print(annot["category_id"])
        #print(clean_cats[:27][annot["category_id"]])
        bbox = bboxarea(annot["bbox"])
        #print("bbox",bbox)
        area = annot["area"]
        #print("precalc area",area)
        
        imgdata = dicts[z][annot["image_id"]] #recover data from imageid+metadata table
        imgsize = imgdata[1]*imgdata[2] #width x height

        area_values[category].append(area/bbox)
        size_ratio_values[category].append(area/imgsize)

        continue #we don't need that right now
        i+=1
        for mask in annot["segmentation"]:
            xs, ys = [], []

            for v in range(len(mask)):
                if v%2==0:
                    xs.append(mask[v])
                else:
                    ys.append(mask[v])
            xs.append(xs[0])
            ys.append(ys[0])
            '''
            plt.figure()
            plt.plot(xs, ys) 
            plt.axis('scaled')
            plt.show()
            '''
            if len(xs)==len(ys):
                print("shoelace",shoelace_formula(xs,ys))
        print()
        #if i > 15: break
    mean_area_ratios = [np.mean(v) for k,v in area_values.items()]
    std_area_ratios = [np.std(v) for k,v in area_values.items()]

    plt.bar(clean_cats[:27],mean_area_ratios,yerr=std_area_ratios)
    plt.xticks(range(len(D)), labels = clean_cats[:27], rotation=90)
    plt.xlim(-0.5,upto-.5)
    plt.title("Bounding box fullness per category. "+str(info))
    plt.show()
    
    mean_area_ratios = [np.mean(v) for k,v in size_ratio_values.items()]
    std_area_ratios = [np.std(v) for k,v in size_ratio_values.items()]

    plt.bar(clean_cats[:27],mean_area_ratios,yerr=std_area_ratios,color="#007700")
    plt.xticks(range(len(D)), labels = clean_cats[:27], rotation=90)
    plt.xlim(-0.5,upto-.5)
    plt.title("Relative mask area per category. "+str(info))
    plt.show()